# Cadenas de Markov y Aplicaciones (bog-2028838-1-2026-03)
## Tarea 2 - Movimiento de un Caballo en el Tablero de Ajedrez

**Integrantes:** Nicolás Betancur Sánchez, Andrés Felipe Gómez y Germán Camilo Rodríguez

---

### Planteamiento General del Problema

Considere un tablero de ajedrez estándar de $8 \times 8$.
- Las **columnas** del tablero se identifican mediante las letras $a, b, c, d, e, f, g, h$.
- Las **filas** del tablero se identifican mediante los números $1, 2, 3, 4, 5, 6, 7, 8$.
- Por tanto, cada casilla del tablero queda unívocamente identificada mediante una letra y un número:
$$\text{Casilla } = c\,r, \quad c \in \{a, b, c, d, e, f, g, h\}, \; r \in \{1, 2, 3, 4, 5, 6, 7, 8\}$$

Considere un caballo que se desplaza aleatoriamente sobre el tablero, asumiendo que no hay otras piezas. En cada turno, el caballo realiza un movimiento legal usual de ajedrez (movimiento en "L": avanza dos casillas en una dirección recta ortogonal y una en ángulo recto, o viceversa). En cada paso, el caballo elige **uniformemente al azar** una de las casillas legales a las que puede desplazarse desde su posición actual.

Sea $X_n$ la posición del caballo en el $n$-ésimo movimiento, para $n = 0, 1, 2, \dots$.
El proceso estocástico $X = (X_n)_{n \ge 0}$ constituye una **Cadena de Markov** a tiempo discreto con espacio de estados finito.


---
## Ejercicio 1. Construcción de la Cadena de Markov

### Literal 1.a) Descripción del espacio de estados $S$
> **Enunciado:** Describa el espacio de estados $S$ del proceso $(X_n)_{n \ge 0}$. ¿Cuántos estados tiene?

El espacio de estados $S$ corresponde a la totalidad de las casillas del tablero de ajedrez sobre las cuales puede ubicarse el caballo. Formalmente, se define como el producto cartesiano del conjunto de letras de las columnas y el conjunto de números de las filas:

$$S = \{a, b, c, d, e, f, g, h\} \times \{1, 2, 3, 4, 5, 6, 7, 8\}$$

$$S = \{a1, a2, \dots, a8, b1, b2, \dots, b8, \dots, h1, h2, \dots, h8\}$$

Dado que el tablero de ajedrez estándar consta de 8 columnas y 8 filas:
$$|S| = 8 \times 8 = 64 \text{ estados}$$

A continuación se genera y verifica computacionalmente el espacio de estados $S$.


In [1]:
# ==============================================================================
# LITERAL 1.a: Espacio de estados S y su cardinalidad |S|
# ==============================================================================

# Definición de las 8 columnas y las 8 filas del tablero
columnas = ["a", "b", "c", "d", "e", "f", "g", "h"]
filas = [1, 2, 3, 4, 5, 6, 7, 8]

# Construcción exhaustiva del espacio de estados S
S = [f"{col}{fila}" for col in columnas for fila in filas]

# Verificación de la cardinalidad
print("=" * 60)
print("LITERAL 1.a: ESPACIO DE ESTADOS S")
print("=" * 60)
print(f"Total de estados en el espacio: |S| = {len(S)}")
print(f"Primeros 8 estados (columna a): {S[:8]}")
print(f"Últimos 8 estados (columna h):  {S[-8:]}")


LITERAL 1.a: ESPACIO DE ESTADOS S
Total de estados en el espacio: |S| = 64
Primeros 8 estados (columna a): ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8']
Últimos 8 estados (columna h):  ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8']


---
### Literal 1.b) Determinación de $N(x)$ y $|N(x)|$ para cada estado $x \in S$
> **Enunciado:** Para un estado $x \in S$, defina:
> $$N(x) = \{y \in S : \text{el caballo puede pasar de } x \text{ a } y \text{ en un solo movimiento}\}$$
> $$|N(x)| = \operatorname{cardinal}(N(x))$$
> Determine $N(x)$ y $|N(x)|$ para cada $x \in S$. Presente los resultados en una tabla que contenga $x$, $N(x)$, $|N(x)|$.

Un caballo situado en una casilla con coordenadas $(\text{columna}, \text{fila})$ realiza desplazamientos en "L". En un plano cartesiano discreto, las traslaciones permitidas son los 8 pares coordenados:
$$(\Delta c, \Delta f) \in \{(\pm 2, \pm 1), (\pm 1, \pm 2)\}$$

Para que una casilla destino $y = (c + \Delta c, f + \Delta f)$ pertenezca a $N(x)$, debe estar estrictamente dentro de los límites del tablero:
$$0 \le c + \Delta c \le 7 \quad \text{y} \quad 0 \le f + \Delta f \le 7$$

Dependiendo de la cercanía a los bordes y esquinas, el número de movimientos legales $|N(x)|$ toma los valores:
- **2 movimientos:** En las 4 esquinas ($a1, a8, h1, h8$).
- **3 movimientos:** En las casillas adyacentes a las esquinas a lo largo del borde.
- **4 movimientos:** En casillas del borde y posiciones sub-esquina.
- **6 movimientos:** En casillas intermedias adyacentes al borde interior.
- **8 movimientos:** En la región central del tablero ($16$ casillas: filas 3 a 6 y columnas $c$ a $f$).

A continuación se calcula $N(x)$ y $|N(x)|$ para las 64 casillas y se presenta la tabla requerida.


In [2]:
# ==============================================================================
# LITERAL 1.b: Determinación de N(x) y |N(x)| para todo x en S
# ==============================================================================

# Vector para mapear el índice de columna (0-7) a letra ('a'-'h')
conv = ["a", "b", "c", "d", "e", "f", "g", "h"]

# N[c][f]: almacenará las tuplas (nc, nf) de casillas accesibles desde (c, f)
# car_N[c][f]: almacenará la cardinalidad |N(x)| de cada casilla (c, f)
N = [[[] for _ in range(8)] for _ in range(8)]
car_N = [[0 for _ in range(8)] for _ in range(8)]

def to_chess(c, f):
    """Convierte índices (0-7, 0-7) a notación algebraica de ajedrez (ej. 'a1')."""
    return f"{conv[c]}{f + 1}"

# Los 8 posibles saltos en 'L' del caballo: (delta_columna, delta_fila)
saltos_caballo = [
    (-2, -1), (-2,  1),
    (-1, -2), (-1,  2),
    ( 1, -2), ( 1,  2),
    ( 2, -1), ( 2,  1)
]

# Exploramos exhaustivamente cada una de las 64 casillas del tablero
for c in range(8):
    for f in range(8):
        for dc, df in saltos_caballo:
            nc, nf = c + dc, f + df
            # Validación de fronteras del tablero 8x8
            if 0 <= nc < 8 and 0 <= nf < 8:
                N[c][f].append((nc, nf))
                car_N[c][f] += 1

# Presentación estructurada de la tabla: x, N(x), |N(x)|
print("=" * 80)
print(f"{'Casilla (x)':<12} | {'|N(x)|':<6} | {'Vecinos accesibles N(x)'}")
print("=" * 80)

for c in range(8):
    for f in range(8):
        casilla = to_chess(c, f)
        cardinal = car_N[c][f]
        vecinos_str = ", ".join(to_chess(nc, nf) for nc, nf in N[c][f])
        print(f"{casilla:<12} | {cardinal:<6} | {vecinos_str}")


Casilla (x)  | |N(x)| | Vecinos accesibles N(x)
a1           | 2      | b3, c2
a2           | 3      | b4, c1, c3
a3           | 4      | b1, b5, c2, c4
a4           | 4      | b2, b6, c3, c5
a5           | 4      | b3, b7, c4, c6
a6           | 4      | b4, b8, c5, c7
a7           | 3      | b5, c6, c8
a8           | 2      | b6, c7
b1           | 3      | a3, c3, d2
b2           | 4      | a4, c4, d1, d3
b3           | 6      | a1, a5, c1, c5, d2, d4
b4           | 6      | a2, a6, c2, c6, d3, d5
b5           | 6      | a3, a7, c3, c7, d4, d6
b6           | 6      | a4, a8, c4, c8, d5, d7
b7           | 4      | a5, c5, d6, d8
b8           | 3      | a6, c6, d7
c1           | 4      | a2, b3, d3, e2
c2           | 6      | a1, a3, b4, d4, e1, e3
c3           | 8      | a2, a4, b1, b5, d1, d5, e2, e4
c4           | 8      | a3, a5, b2, b6, d2, d6, e3, e5
c5           | 8      | a4, a6, b3, b7, d3, d7, e4, e6
c6           | 8      | a5, a7, b4, b8, d4, d8, e5, e7
c7           | 6      

---
### Literal 1.c) Cálculo de las probabilidades de transición $P_{xy}$
> **Enunciado:** Teniendo en cuenta que el caballo elige moverse uniformemente entre todos los movimientos legales disponibles desde $x$, calcule las probabilidades de transición:
> $$P_{xy} = P(X_{n+1} = y \mid X_n = x) = \begin{cases} \dfrac{1}{|N(x)|}, & y \in N(x) \\[6pt] 0, & y \notin N(x) \end{cases}$$
> **Ejemplo:** $P_{a1, y} = \begin{cases} \dfrac{1}{2}, & y = b3, c2 \\[6pt] 0, & \text{caso contrario.} \end{cases}$

Bajo la hipótesis del enunciado, el caballo selecciona su próximo destino según una distribución uniforme discreta sobre el conjunto de movimientos legales $N(x)$. Como consecuencia:
1. Para cada casilla destino legal $y \in N(x)$, la probabilidad de transición es idéntica e igual a $\frac{1}{|N(x)|}$.
2. Para cualquier otra casilla $y \notin N(x)$, la probabilidad es nula ($0$).
3. La suma de probabilidades sobre todos los estados de llegada es unitaria:
$$\sum_{y \in S} P_{xy} = \sum_{y \in N(x)} \frac{1}{|N(x)|} = |N(x)| \times \frac{1}{|N(x)|} = 1, \quad \forall x \in S$$

A continuación se implementa la función evaluadora $P_{xy}$ y se verifica con el ejemplo oficial de $a1$.


In [3]:
# ==============================================================================
# LITERAL 1.c: Función de cálculo de las probabilidades de transición P_xy
# ==============================================================================

def P_xy(x_nombre, y_nombre):
    """
    Calcula P_xy = P(X_{n+1} = y | X_n = x) de forma exacta.
    Retorna (probabilidad_flotante, representacion_fraccionaria).
    """
    cx, fx = conv.index(x_nombre[0]), int(x_nombre[1]) - 1
    cy, fy = conv.index(y_nombre[0]), int(y_nombre[1]) - 1
    
    if (cy, fy) in N[cx][fx]:
        grado = car_N[cx][fx]
        return 1.0 / grado, f"1/{grado}"
    else:
        return 0.0, "0"

# Verificación directa con el ejemplo del enunciado: Casilla a1
print("=" * 60)
print("LITERAL 1.c: VERIFICACIÓN CON EL EJEMPLO DE LA CASILLA a1")
print("=" * 60)
ejemplos_y = ["b3", "c2", "a2", "b1", "d4", "h8"]
for dest in ejemplos_y:
    prob_val, prob_str = P_xy("a1", dest)
    print(f"P(X_{{n+1}} = {dest} | X_n = a1) = {prob_str:<4} ({prob_val:.4f})")


LITERAL 1.c: VERIFICACIÓN CON EL EJEMPLO DE LA CASILLA a1
P(X_{n+1} = b3 | X_n = a1) = 1/2  (0.5000)
P(X_{n+1} = c2 | X_n = a1) = 1/2  (0.5000)
P(X_{n+1} = a2 | X_n = a1) = 0    (0.0000)
P(X_{n+1} = b1 | X_n = a1) = 0    (0.0000)
P(X_{n+1} = d4 | X_n = a1) = 0    (0.0000)
P(X_{n+1} = h8 | X_n = a1) = 0    (0.0000)


---
### Literal 1.d) Presentación de las probabilidades $P_{h8, y}$, $P_{g1, y}$, $P_{b7, y}$, $P_{f7, y}$, $P_{e5, y}$
> **Enunciado:** Presente las probabilidades de transición $P_{h8, y}$, $P_{g1, y}$, $P_{b7, y}$, $P_{f7, y}$, $P_{e5, y}$.

Las cinco casillas seleccionadas ilustran con precisión los diferentes grados de libertad del caballo en el tablero:

1. **Casilla de esquina $h8$ ($|N(h8)| = 2$):**
   Sus únicos movimientos legales son $N(h8) = \{f7, g6\}$. Por ende:
   $$P_{h8, y} = \begin{cases} \dfrac{1}{2} = 0.5, & y \in \{f7, g6\} \\[6pt] 0, & \text{caso contrario.} \end{cases}$$

2. **Casilla de borde exterior $g1$ ($|N(g1)| = 3$):**
   Sus movimientos legales son $N(g1) = \{e2, f3, h3\}$. Por ende:
   $$P_{g1, y} = \begin{cases} \dfrac{1}{3} \approx 0.3333, & y \in \{e2, f3, h3\} \\[6pt] 0, & \text{caso contrario.} \end{cases}$$

3. **Casilla de sub-borde $b7$ ($|N(b7)| = 4$):**
   Sus movimientos legales son $N(b7) = \{a5, c5, d6, d8\}$. Por ende:
   $$P_{b7, y} = \begin{cases} \dfrac{1}{4} = 0.25, & y \in \{a5, c5, d6, d8\} \\[6pt] 0, & \text{caso contrario.} \end{cases}$$

4. **Casilla semi-interior $f7$ ($|N(f7)| = 6$):**
   Sus movimientos legales son $N(f7) = \{d6, d8, e5, g5, h6, h8\}$. Por ende:
   $$P_{f7, y} = \begin{cases} \dfrac{1}{6} \approx 0.1667, & y \in \{d6, d8, e5, g5, h6, h8\} \\[6pt] 0, & \text{caso contrario.} \end{cases}$$

5. **Casilla del centro del tablero $e5$ ($|N(e5)| = 8$):**
   Sus movimientos legales son $N(e5) = \{c4, c6, d3, d7, f3, f7, g4, g6\}$. Por ende:
   $$P_{e5, y} = \begin{cases} \dfrac{1}{8} = 0.125, & y \in \{c4, c6, d3, d7, f3, f7, g4, g6\} \\[6pt] 0, & \text{caso contrario.} \end{cases}$$

A continuación se imprimen programáticamente todas estas distribuciones de probabilidad.


In [4]:
# ==============================================================================
# LITERAL 1.d: Presentación programática de P_{h8,y}, P_{g1,y}, P_{b7,y}, P_{f7,y}, P_{e5,y}
# ==============================================================================

casillas_d = ["h8", "g1", "b7", "f7", "e5"]

print("=" * 75)
print("LITERAL 1.d: PROBABILIDADES DE TRANSICIÓN ESPECÍFICAS SOLICITADAS")
print("=" * 75)

for cas in casillas_d:
    c_idx = conv.index(cas[0])
    f_idx = int(cas[1]) - 1
    grado = car_N[c_idx][f_idx]
    vecinos = [to_chess(nc, nf) for nc, nf in N[c_idx][f_idx]]
    
    print(f"\n● Casilla x = {cas}  -->  Grado |N({cas})| = {grado} movimientos posibles")
    print(f"  Definición por partes:")
    print(f"    P_{{{cas}, y}} = 1/{grado} ({1.0/grado:.4f})  para  y ∈ {{{', '.join(vecinos)}}}")
    print(f"    P_{{{cas}, y}} = 0             para  y ∉ {{{', '.join(vecinos)}}}")
    print(f"  Detalle de probabilidades no nulas:")
    for v in vecinos:
        p_val, p_str = P_xy(cas, v)
        print(f"    P({cas} -> {v}) = {p_str} = {p_val:.4f}")


LITERAL 1.d: PROBABILIDADES DE TRANSICIÓN ESPECÍFICAS SOLICITADAS

● Casilla x = h8  -->  Grado |N(h8)| = 2 movimientos posibles
  Definición por partes:
    P_{h8, y} = 1/2 (0.5000)  para  y ∈ {f7, g6}
    P_{h8, y} = 0             para  y ∉ {f7, g6}
  Detalle de probabilidades no nulas:
    P(h8 -> f7) = 1/2 = 0.5000
    P(h8 -> g6) = 1/2 = 0.5000

● Casilla x = g1  -->  Grado |N(g1)| = 3 movimientos posibles
  Definición por partes:
    P_{g1, y} = 1/3 (0.3333)  para  y ∈ {e2, f3, h3}
    P_{g1, y} = 0             para  y ∉ {e2, f3, h3}
  Detalle de probabilidades no nulas:
    P(g1 -> e2) = 1/3 = 0.3333
    P(g1 -> f3) = 1/3 = 0.3333
    P(g1 -> h3) = 1/3 = 0.3333

● Casilla x = b7  -->  Grado |N(b7)| = 4 movimientos posibles
  Definición por partes:
    P_{b7, y} = 1/4 (0.2500)  para  y ∈ {a5, c5, d6, d8}
    P_{b7, y} = 0             para  y ∉ {a5, c5, d6, d8}
  Detalle de probabilidades no nulas:
    P(b7 -> a5) = 1/4 = 0.2500
    P(b7 -> c5) = 1/4 = 0.2500
    P(b7 -> d6) = 1/4

---
### Literal 1.e) Construcción de la matriz de transición $P$ y filas de $g1$ y $h6$
> **Enunciado:** Fije un orden para las 64 casillas del tablero. Por ejemplo, puede utilizar $a1, a2, \dots, a8, b1, b2, \dots, b8, \dots, h1, \dots, h8$.
> Construya la matriz de transición $P = (P_{xy})_{x,y \in S}$.
> La matriz puede ser construida utilizando un programa, incluya en el informe el código utilizado (o el enlace al código usado). No es necesario escribir todas sus entradas.
> Incluya en el informe, al menos, las filas de la matriz correspondientes a las casillas $g1, h6$.

#### 1. Fijación del orden canónico
Se adopta el orden lexicográfico estándar indicado en el enunciado:
$$a1, a2, \dots, a8, \quad b1, b2, \dots, b8, \quad \dots, \quad h1, h2, \dots, h8$$

Con este ordenamiento, cada casilla $(c, f)$ donde $c \in \{0, 1, \dots, 7\}$ (asociado a $a, \dots, h$) y $f \in \{0, 1, \dots, 7\}$ (asociado a $1, \dots, 8$) se mapea a un índice entero único entre $0$ y $63$:
$$\text{índice}(c, f) = c \times 8 + f$$

Por ejemplo:
- $a1 \to 0 \times 8 + 0 = 0$
- $a8 \to 0 \times 8 + 7 = 7$
- $g1 \to 6 \times 8 + 0 = 48$
- $h6 \to 7 \times 8 + 5 = 61$
- $h8 \to 7 \times 8 + 7 = 63$

#### 2. Referencia al código fuente utilizado
El algoritmo para generar y estructurar la matriz de transición $P$ fue implementado en los scripts del proyecto:
- Script principal en la raíz: [`1b.py`](file:///d:/UNAL/C.C/Cadenas%20de%20Markov/Tareas_Cadenas_de_Markov/1b.py)
- Script en la carpeta de la tarea: [`Tareas/Tarea2/1b_solucion_andres.py`](file:///d:/UNAL/C.C/Cadenas%20de%20Markov/Tareas_Cadenas_de_Markov/Tareas/Tarea2/1b_solucion_andres.py)

A continuación se ejecuta dicho código en este notebook, se verifican las propiedades estocásticas de la matriz $P$ y se imprimen las filas de las casillas $g1$ y $h6$ solicitadas.


In [5]:
# ==============================================================================
# LITERAL 1.e: Construcción de la matriz de transición P (64x64)
# (Código referenciado de 1b.py y Tareas/Tarea2/1b_solucion_andres.py)
# ==============================================================================

# Inicialización de la matriz de transición P (64 x 64)
P = [[0.0 for _ in range(64)] for _ in range(64)]

# Construcción de las 64 filas de P
for c in range(8):
    for f in range(8):
        x = c * 8 + f                     # Índice de la fila correspondiente a (c, f)
        grado = car_N[c][f]
        for nc, nf in N[c][f]:
            y = nc * 8 + nf               # Índice de la columna correspondiente al vecino
            P[x][y] = 1.0 / grado         # Asignación de la probabilidad uniforme

# ------------------------------------------------------------------------------
# Verificaciones formales de la matriz de transición P
# ------------------------------------------------------------------------------
suma_filas = [sum(P[i]) for i in range(64)]
filas_estocasticas = all(abs(s - 1.0) < 1e-9 for s in suma_filas)
diagonal_nula = all(P[i][i] == 0.0 for i in range(64))

print("=" * 70)
print("LITERAL 1.e: VERIFICACIONES DE LA MATRIZ DE TRANSICIÓN P")
print("=" * 70)
print(f"Dimensiones de la matriz: {len(P)} x {len(P[0])}")
print(f"¿Todas las 64 filas suman 1.0? -> {filas_estocasticas}")
print(f"¿La diagonal principal es completamente 0? -> {diagonal_nula}")

# ------------------------------------------------------------------------------
# Función para visualizar las filas de la matriz
# ------------------------------------------------------------------------------
def mostrar_fila_completa(casilla):
    col = conv.index(casilla[0])
    fil = int(casilla[1]) - 1
    idx = col * 8 + fil
    fila = P[idx]
    
    print(f"\n" + "-" * 70)
    print(f"FILA DE LA MATRIZ P PARA LA CASILLA '{casilla}' (Índice de fila x = {idx})")
    print("-" * 70)
    print("Transiciones con probabilidad positiva (P_xy > 0):")
    for y_idx in range(64):
        if fila[y_idx] > 0:
            c_dest, f_dest = divmod(y_idx, 8)
            y_nombre = to_chess(c_dest, f_dest)
            print(f"  P({casilla} -> {y_nombre}) = P[{idx:>2}][{y_idx:>2}] = {fila[y_idx]:.4f}")
            
    print(f"\nRepresentación vectorial completa P[{idx}, :] (64 columnas):")
    for bloque in range(8):
        etiqueta = f"{conv[bloque]}1..{conv[bloque]}8"
        valores = " ".join(f"{fila[bloque*8 + k]:.2f}" for k in range(8))
        print(f"  [{etiqueta}]: {valores}")

# Mostrar las dos filas solicitadas en la tarea
mostrar_fila_completa("g1")
mostrar_fila_completa("h6")


LITERAL 1.e: VERIFICACIONES DE LA MATRIZ DE TRANSICIÓN P
Dimensiones de la matriz: 64 x 64
¿Todas las 64 filas suman 1.0? -> True
¿La diagonal principal es completamente 0? -> True

----------------------------------------------------------------------
FILA DE LA MATRIZ P PARA LA CASILLA 'g1' (Índice de fila x = 48)
----------------------------------------------------------------------
Transiciones con probabilidad positiva (P_xy > 0):
  P(g1 -> e2) = P[48][33] = 0.3333
  P(g1 -> f3) = P[48][42] = 0.3333
  P(g1 -> h3) = P[48][58] = 0.3333

Representación vectorial completa P[48, :] (64 columnas):
  [a1..a8]: 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  [b1..b8]: 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  [c1..c8]: 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  [d1..d8]: 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  [e1..e8]: 0.00 0.33 0.00 0.00 0.00 0.00 0.00 0.00
  [f1..f8]: 0.00 0.00 0.33 0.00 0.00 0.00 0.00 0.00
  [g1..g8]: 0.00 0.00 0.00 0.00 0.00 0.00 0.00 0.00
  [h1..h8]: 0.00 0.00 0.33 0.00 

---
### Literal 1.f) ¿La matriz $P$ es simétrica? Justifique su respuesta.

Para verificar que la matriz de transición $P \in \mathbb{R}^{64 \times 64}$ sea simétrica, debemos verificar que $P_{ij} = P_{ji}$ para todo par de estados (casillas) $i$ y $j$.<br>

Sea $S = \{1, 2, \dots, 64\}$ el conjunto de estados que representan las casillas del tablero. Para cada casilla $x \in S$, denotamos como $N(x)$ el conjunto de estados a los que se puede llegar en un movimiento desde la casilla $x \in S$, y $|N(x)| = d(x)$ el número de dichos estados.<br>

Dado que el movimiento se distribuye uniforme entre los elementos de $N(x)$, la probabilidad de pasar del estado $i$ al estado $j$ en un paso es:<br>

$$P_{ij} = \begin{cases}
\dfrac{1}{|N(i)|} & \text{si } j \text{ es un movimiento legal desde } i, \\[6pt]
0 & \text{en caso contrario,}
\end{cases}$$

Como $P_{ij} = P_{ji}$ para todo $i$ y $j$, para casillas con movimientos legales mutuos:
$$\frac{1}{d(i)} = \frac{1}{d(j)} \iff d(i) = d(j) \quad (\text{es decir, } |N(i)| = |N(j)|)$$

Veamos que, según lo que ya hemos calculado anteriormente, por contraejemplo, la matriz de transiciones no es simétrica:<br><br>
$N(a1) = \{c2, b3\}$ y $N(b3) = \{d2, d4, a1, c1, a5, c5\}$. Como $|N(a1)| = 2$ y $|N(b3)| = 6$, la probabilidad sobre cada uno para llegar a otra casilla legal, con distribución uniforme, es $P_{a1, j} = \frac{1}{2}$ y $P_{b3, j} = \frac{1}{6}$.<br>
Dado que $P_{a1, b3} = \frac{1}{2} \neq \frac{1}{6} = P_{b3, a1}$, la matriz no cumple la propiedad $P = P^T$, no es simétrica.


---
# Ejercicio 2. Cálculo de Probabilidades en la Cadena de Markov

> **Hipótesis General:** Suponga ahora que el caballo inicia en la casilla $a1$, es decir:
> $$X_0 = a1 \implies P(X_0 = a1) = 1$$

---
### 2.1) Cálculo de $P(X_1 = c2)$


#### Desarrollo Matemático:
Dado que el proceso se encuentra inicialmente en el estado $X_0 = a1$, por la definición de la cadena de Markov y probabilidad condicional:
$$P(X_1 = c2) = P(X_1 = c2 \mid X_0 = a1) \cdot P(X_0 = a1) = P_{a1, c2} \cdot 1 = P_{a1, c2}$$

Desde la casilla de esquina $a1$, el conjunto de movimientos legales es:
$$N(a1) = \{b3, c2\}, \quad \text{con } |N(a1)| = 2$$

Como la casilla $c2 \in N(a1)$, el caballo elige uniformemente entre sus 2 movimientos legales disponibles:
$$P(X_1 = c2) = \frac{1}{|N(a1)|} = \frac{1}{2} = 0.5$$


In [6]:
# ==============================================================================
# PUNTO 2.1: Cálculo de P(X_1 = c2) iniciando en a1
# ==============================================================================

# Estado inicial
x0 = "a1"
c_x0 = conv.index(x0[0])
f_x0 = int(x0[1]) - 1

vecinos_a1 = [to_chess(nc, nf) for nc, nf in N[c_x0][f_x0]]
grado_a1 = car_N[c_x0][f_x0]

print("=" * 70)
print("PUNTO 2.1: CÁLCULO DE P(X_1 = c2)")
print("=" * 70)
print(f"Estado inicial: X_0 = {x0}")
print(f"Vecinos de {x0}: N({x0}) = {vecinos_a1}")
print(f"Número de movimientos legales: |N({x0})| = {grado_a1}")

destino = "c2"
esta_en_vecinos = destino in vecinos_a1
print(f"¿El destino '{destino}' es alcanzable desde '{x0}' (c2 ∈ N(a1))? -> {esta_en_vecinos}")

if esta_en_vecinos:
    p_x1_c2 = 1.0 / grado_a1
    print(f"Probabilidad: P(X_1 = {destino}) = 1/|N({x0})| = 1/{grado_a1} = {p_x1_c2:.4f}")


PUNTO 2.1: CÁLCULO DE P(X_1 = c2)
Estado inicial: X_0 = a1
Vecinos de a1: N(a1) = ['b3', 'c2']
Número de movimientos legales: |N(a1)| = 2
¿El destino 'c2' es alcanzable desde 'a1' (c2 ∈ N(a1))? -> True
Probabilidad: P(X_1 = c2) = 1/|N(a1)| = 1/2 = 0.5000


---
### 2.2) Cálculo de $P(X_1 = c2, X_2 = d4, X_3 = e6, X_4 = c7)$


#### Desarrollo Matemático:
Por la regla de la multiplicación de probabilidades condicionales (regla de la cadena) y la propiedad de Markov (el siguiente estado depende únicamente del estado actual e independientemente de los anteriores):

$$P(X_1 = c2, X_2 = d4, X_3 = e6, X_4 = c7) = P(X_1 = c2 \mid X_0 = a1) \cdot P(X_2 = d4 \mid X_1 = c2) \cdot P(X_3 = e6 \mid X_2 = d4) \cdot P(X_4 = c7 \mid X_3 = e6)$$
$$= P_{a1, c2} \cdot P_{c2, d4} \cdot P_{d4, e6} \cdot P_{e6, c7}$$

Verificamos la legalidad y probabilidad de cada transición individual:
1. **Paso $0 \to 1$ ($a1 \to c2$):**
   $N(a1) = \{b3, c2\}$, con $|N(a1)| = 2$.
   Como $c2 \in N(a1) \implies P_{a1, c2} = \dfrac{1}{2}$.

2. **Paso $1 \to 2$ ($c2 \to d4$):**
   $N(c2) = \{a1, a3, b4, d4, e1, e3\}$, con $|N(c2)| = 6$.
   Como $d4 \in N(c2) \implies P_{c2, d4} = \dfrac{1}{6}$.

3. **Paso $2 \to 3$ ($d4 \to e6$):**
   $N(d4) = \{b3, b5, c2, c6, e2, e6, f3, f5\}$, con $|N(d4)| = 8$.
   Como $e6 \in N(d4) \implies P_{d4, e6} = \dfrac{1}{8}$.

4. **Paso $3 \to 4$ ($e6 \to c7$):**
   $N(e6) = \{c5, c7, d4, d8, f4, f8, g5, g7\}$, con $|N(e6)| = 8$.
   Como $c7 \in N(e6) \implies P_{e6, c7} = \dfrac{1}{8}$.

Multiplicando las probabilidades de cada paso:
$$P(X_1 = c2, X_2 = d4, X_3 = e6, X_4 = c7) = \frac{1}{2} \times \frac{1}{6} \times \frac{1}{8} \times \frac{1}{8} = \frac{1}{768} \approx 0.00130208$$


In [7]:
# ==============================================================================
# PUNTO 2.2: Cálculo paso a paso de P(X_1=c2, X_2=d4, X_3=e6, X_4=c7)
# ==============================================================================
from fractions import Fraction

trayectoria_22 = ["a1", "c2", "d4", "e6", "c7"]

print("=" * 70)
print("PUNTO 2.2: CÁLCULO PASO A PASO DE LA TRAYECTORIA")
print("=" * 70)

prob_acumulada = Fraction(1, 1)

for paso in range(len(trayectoria_22) - 1):
    origen = trayectoria_22[paso]
    destino = trayectoria_22[paso + 1]
    
    c_orig, f_orig = conv.index(origen[0]), int(origen[1]) - 1
    vecinos_orig = [to_chess(nc, nf) for nc, nf in N[c_orig][f_orig]]
    grado_orig = car_N[c_orig][f_orig]
    
    es_legal = destino in vecinos_orig
    p_paso = Fraction(1, grado_orig) if es_legal else Fraction(0, 1)
    prob_acumulada *= p_paso
    
    print(f"\nPaso {paso} -> {paso+1}: Transición de {origen} hacia {destino}")
    print(f"  N({origen}) = {vecinos_orig}")
    print(f"  |N({origen})| = {grado_orig}")
    print(f"  ¿{destino} ∈ N({origen})? -> {es_legal}")
    print(f"  P(X_{paso+1} = {destino} | X_{paso} = {origen}) = {p_paso} ({float(p_paso):.4f})")

print("\n" + "-" * 70)
print(f"Probabilidad total conjunta P(X_1=c2, X_2=d4, X_3=e6, X_4=c7):")
print(f"  Fracción exacta: {prob_acumulada}")
print(f"  Valor decimal:   {float(prob_acumulada):.8f}")


PUNTO 2.2: CÁLCULO PASO A PASO DE LA TRAYECTORIA

Paso 0 -> 1: Transición de a1 hacia c2
  N(a1) = ['b3', 'c2']
  |N(a1)| = 2
  ¿c2 ∈ N(a1)? -> True
  P(X_1 = c2 | X_0 = a1) = 1/2 (0.5000)

Paso 1 -> 2: Transición de c2 hacia d4
  N(c2) = ['a1', 'a3', 'b4', 'd4', 'e1', 'e3']
  |N(c2)| = 6
  ¿d4 ∈ N(c2)? -> True
  P(X_2 = d4 | X_1 = c2) = 1/6 (0.1667)

Paso 2 -> 3: Transición de d4 hacia e6
  N(d4) = ['b3', 'b5', 'c2', 'c6', 'e2', 'e6', 'f3', 'f5']
  |N(d4)| = 8
  ¿e6 ∈ N(d4)? -> True
  P(X_3 = e6 | X_2 = d4) = 1/8 (0.1250)

Paso 3 -> 4: Transición de e6 hacia c7
  N(e6) = ['c5', 'c7', 'd4', 'd8', 'f4', 'f8', 'g5', 'g7']
  |N(e6)| = 8
  ¿c7 ∈ N(e6)? -> True
  P(X_4 = c7 | X_3 = e6) = 1/8 (0.1250)

----------------------------------------------------------------------
Probabilidad total conjunta P(X_1=c2, X_2=d4, X_3=e6, X_4=c7):
  Fracción exacta: 1/768
  Valor decimal:   0.00130208


---
### 2.3) Cálculo de $P(X_3 = e2, X_2 = d4 \mid X_1 = b3)$


#### Desarrollo Matemático:
1. **Factibilidad del evento condicionante:**
   Iniciando en $X_0 = a1$, tenemos que $N(a1) = \{b3, c2\}$. Puesto que $b3 \in N(a1)$, la probabilidad del evento condicionante es estrictamente positiva:
   $$P(X_1 = b3) = \frac{1}{|N(a1)|} = \frac{1}{2} > 0$$

2. **Aplicación de la regla de la cadena y propiedad de Markov:**
   Por la definición de probabilidad condicional:
   $$P(X_3 = e2, X_2 = d4 \mid X_1 = b3) = P(X_2 = d4 \mid X_1 = b3) \cdot P(X_3 = e2 \mid X_2 = d4, X_1 = b3)$$

   Al ser $(X_n)$ una **Cadena de Markov**, el proceso carece de memoria sobre el pasado histórico: dado el estado presente $X_2 = d4$, la probabilidad del estado futuro $X_3 = e2$ es condicionalmente independiente del estado anterior $X_1 = b3$:
   $$P(X_3 = e2 \mid X_2 = d4, X_1 = b3) = P(X_3 = e2 \mid X_2 = d4) = P_{d4, e2}$$

   Por consiguiente, la expresión se reduce al producto de las probabilidades de transición de un paso:
   $$P(X_3 = e2, X_2 = d4 \mid X_1 = b3) = P_{b3, d4} \cdot P_{d4, e2}$$

3. **Cálculo de cada probabilidad de transición:**
   - **Paso $1 \to 2$ ($b3 \to d4$):**
     $N(b3) = \{a1, a5, c1, c5, d2, d4\}$, con $|N(b3)| = 6$.
     Como $d4 \in N(b3) \implies P_{b3, d4} = \dfrac{1}{6}$.
   - **Paso $2 \to 3$ ($d4 \to e2$):**
     $N(d4) = \{b3, b5, c2, c6, e2, e6, f3, f5\}$, con $|N(d4)| = 8$.
     Como $e2 \in N(d4) \implies P_{d4, e2} = \dfrac{1}{8}$.

Multiplicando ambas probabilidades:
$$P(X_3 = e2, X_2 = d4 \mid X_1 = b3) = \frac{1}{6} \times \frac{1}{8} = \frac{1}{48} \approx 0.0208333$$

#### Enfoque equivalente mediante la definición cociente de probabilidad condicional:
$$P(X_3 = e2, X_2 = d4 \mid X_1 = b3) = \frac{P(X_1 = b3, X_2 = d4, X_3 = e2)}{P(X_1 = b3)} = \frac{P_{a1, b3} \cdot P_{b3, d4} \cdot P_{d4, e2}}{P_{a1, b3}} = \frac{\frac{1}{2} \cdot \frac{1}{6} \cdot \frac{1}{8}}{\frac{1}{2}} = \frac{1/96}{1/2} = \frac{1}{48}$$


In [8]:
# ==============================================================================
# PUNTO 2.3: Cálculo paso a paso de P(X_3 = e2, X_2 = d4 | X_1 = b3)
# ==============================================================================

print("=" * 70)
print("PUNTO 2.3: CÁLCULO DE PROBABILIDAD CONDICIONAL")
print("=" * 70)

# Verificación de que X_1 = b3 es posible desde X_0 = a1
vecinos_a1 = [to_chess(nc, nf) for nc, nf in N[conv.index("a")][0]]
print(f"1. Verificación del evento condicionante:")
print(f"   N(a1) = {vecinos_a1}")
print(f"   ¿b3 ∈ N(a1)? -> {'b3' in vecinos_a1}")
p_x1_b3 = Fraction(1, car_N[conv.index("a")][0])
print(f"   P(X_1 = b3 | X_0 = a1) = {p_x1_b3}")

# Paso 1 -> 2: de b3 hacia d4
vecinos_b3 = [to_chess(nc, nf) for nc, nf in N[conv.index("b")][2]]
grado_b3 = car_N[conv.index("b")][2]
print(f"\n2. Transición Paso 1 -> Paso 2 (b3 -> d4):")
print(f"   N(b3) = {vecinos_b3}")
print(f"   |N(b3)| = {grado_b3}")
print(f"   ¿d4 ∈ N(b3)? -> {'d4' in vecinos_b3}")
p_b3_d4 = Fraction(1, grado_b3)
print(f"   P(X_2 = d4 | X_1 = b3) = {p_b3_d4}")

# Paso 2 -> 3: de d4 hacia e2
vecinos_d4 = [to_chess(nc, nf) for nc, nf in N[conv.index("d")][3]]
grado_d4 = car_N[conv.index("d")][3]
print(f"\n3. Transición Paso 2 -> Paso 3 (d4 -> e2):")
print(f"   N(d4) = {vecinos_d4}")
print(f"   |N(d4)| = {grado_d4}")
print(f"   ¿e2 ∈ N(d4)? -> {'e2' in vecinos_d4}")
p_d4_e2 = Fraction(1, grado_d4)
print(f"   P(X_3 = e2 | X_2 = d4) = {p_d4_e2}")

# Probabilidad condicional total
p_cond_total = p_b3_d4 * p_d4_e2

print("\n" + "-" * 70)
print(f"Probabilidad condicional P(X_3 = e2, X_2 = d4 | X_1 = b3):")
print(f"  P_{{b3, d4}} * P_{{d4, e2}} = {p_b3_d4} * {p_d4_e2} = {p_cond_total}")
print(f"  Valor decimal: {float(p_cond_total):.6f}")

# Verificación por cociente de probabilidades
p_interseccion = p_x1_b3 * p_b3_d4 * p_d4_e2
p_verif_cociente = p_interseccion / p_x1_b3
print(f"  Verificación por cociente P(intersección) / P(condición): {p_interseccion} / {p_x1_b3} = {p_verif_cociente}")


PUNTO 2.3: CÁLCULO DE PROBABILIDAD CONDICIONAL
1. Verificación del evento condicionante:
   N(a1) = ['b3', 'c2']
   ¿b3 ∈ N(a1)? -> True
   P(X_1 = b3 | X_0 = a1) = 1/2

2. Transición Paso 1 -> Paso 2 (b3 -> d4):
   N(b3) = ['a1', 'a5', 'c1', 'c5', 'd2', 'd4']
   |N(b3)| = 6
   ¿d4 ∈ N(b3)? -> True
   P(X_2 = d4 | X_1 = b3) = 1/6

3. Transición Paso 2 -> Paso 3 (d4 -> e2):
   N(d4) = ['b3', 'b5', 'c2', 'c6', 'e2', 'e6', 'f3', 'f5']
   |N(d4)| = 8
   ¿e2 ∈ N(d4)? -> True
   P(X_3 = e2 | X_2 = d4) = 1/8

----------------------------------------------------------------------
Probabilidad condicional P(X_3 = e2, X_2 = d4 | X_1 = b3):
  P_{b3, d4} * P_{d4, e2} = 1/6 * 1/8 = 1/48
  Valor decimal: 0.020833
  Verificación por cociente P(intersección) / P(condición): 1/96 / 1/2 = 1/48


---
### 2.4) Cálculo de $P(X_{n+2} = e3 \mid X_n = a3)$
> **Objetivo:** Calcular la probabilidad de que el caballo se encuentre en la casilla $e3$ dos pasos después de encontrarse en la casilla $a3$, para cualquier paso general $n$.

#### Desarrollo Matemático:

1. **Independencia del tiempo $n$ (Homogeneidad temporal):**
   En este proceso estocástico, las reglas de movimiento del caballo no cambian a lo largo del tiempo: sin importar en qué paso $n$ nos encontremos ($n=0, 1, 2, \dots$), desde una casilla dada $x$, las casillas legales accesibles $N(x)$ y las probabilidades de transición hacia ellas son siempre exactamente las mismas.
   Por lo tanto, la probabilidad de desplazarse de la casilla $a3$ a la casilla $e3$ en dos pasos no depende del momento específico $n$ en que el caballo esté en $a3$, sino únicamente del hecho de que transcurren 2 pasos. En consecuencia:
   $$P(X_{n+2} = e3 \mid X_n = a3) = P(X_2 = e3 \mid X_0 = a3)$$

2. **Por probabilidad total:**
   Para desplazarse desde $a3$ en el paso $n$ hasta $e3$ en el paso $n+2$, el caballo debe pasar obligatoriamente por alguna casilla intermedia $k \in S$ en el paso intermedio $n+1$. Por el teorema de **probabilidad total**, particionamos sobre todas las posibles casillas intermedias $k = X_{n+1} \in S$:
   $$P(X_{n+2} = e3 \mid X_n = a3) = \sum_{k \in S} P(X_{n+1} = k, X_{n+2} = e3 \mid X_n = a3)$$

   Utilizando que el proceso es una **Cadena de Markov**, dado que el caballo se encuentra en la casilla intermedia $k$ en el paso $n+1$, la probabilidad de llegar a $e3$ en el paso $n+2$ depende exclusivamente de estar en $k$ y no de haber estado previamente en $a3$:
   $$P(X_{n+2} = e3 \mid X_{n+1} = k, X_n = a3) = P(X_{n+2} = e3 \mid X_{n+1} = k) = P_{k, e3}$$

   Por ende:
   $$P(X_{n+2} = e3 \mid X_n = a3) = \sum_{k \in S} P(X_{n+1} = k \mid X_n = a3) \cdot P(X_{n+2} = e3 \mid X_{n+1} = k) = \sum_{k \in S} P_{a3, k} \cdot P_{k, e3}$$

3. **Identificación de los estados intermedios factibles:**
   Para que un sumando sea estrictamente positivo ($P_{a3, k} \cdot P_{k, e3} > 0$), se deben cumplir simultáneamente dos condiciones:
   - $P_{a3, k} > 0 \iff k \in N(a3)$ (se puede saltar de $a3$ a $k$).
   - $P_{k, e3} > 0 \iff e3 \in N(k) \iff k \in N(e3)$ (se puede saltar de $k$ a $e3$, pues el salto de caballo es bidireccional).

   Por lo tanto, las únicas casillas intermedias $k$ que aportan probabilidad son aquellas en la intersección:
   $$k \in N(a3) \cap N(e3)$$

   Revisemos los vecindarios calculados previamente:
   - $N(a3) = \{b1, b5, c2, c4\}$, con $|N(a3)| = 4$.
   - $N(e3) = \{c2, c4, d1, d5, f1, f5, g2, g4\}$, con $|N(e3)| = 8$.

   La intersección es:
   $$N(a3) \cap N(e3) = \{c2, c4\}$$

4. **Cálculo de los dos caminos posibles de 2 pasos:**
   - **Camino 1 (vía $c2$):** $a3 \to c2 \to e3$
     $$P_{a3, c2} \cdot P_{c2, e3} = \frac{1}{|N(a3)|} \times \frac{1}{|N(c2)|} = \frac{1}{4} \times \frac{1}{6} = \frac{1}{24}$$
   - **Camino 2 (vía $c4$):** $a3 \to c4 \to e3$
     $$P_{a3, c4} \cdot P_{c4, e3} = \frac{1}{|N(a3)|} \times \frac{1}{|N(c4)|} = \frac{1}{4} \times \frac{1}{8} = \frac{1}{32}$$

5. **Suma de probabilidades:**
   $$P(X_{n+2} = e3 \mid X_n = a3) = \frac{1}{24} + \frac{1}{32} = \frac{4 + 3}{96} = \frac{7}{96} \approx 0.07291667$$

*(Nota: En términos matriciales, esta suma corresponde exactamente al producto escalar de la fila $a3$ con la columna $e3$ de la matriz $P$, es decir, la entrada $(P^2)_{a3, e3}$).*


In [9]:
# ==============================================================================
# PUNTO 2.4: Cálculo de P(X_{n+2} = e3 | X_n = a3)
# ==============================================================================

vecinos_a3 = [to_chess(nc, nf) for nc, nf in N[conv.index("a")][2]]
vecinos_e3 = [to_chess(nc, nf) for nc, nf in N[conv.index("e")][2]]

# Intersección de vecinos (casillas intermedias alcanzables)
intermedias = sorted(list(set(vecinos_a3).intersection(set(vecinos_e3))))

print("=" * 70)
print("PUNTO 2.4: CÁLCULO DE P(X_{n+2} = e3 | X_n = a3) EN 2 PASOS")
print("=" * 70)
print(f"Vecinos de a3: N(a3) = {vecinos_a3} (|N(a3)| = {len(vecinos_a3)})")
print(f"Vecinos de e3: N(e3) = {vecinos_e3} (|N(e3)| = {len(vecinos_e3)})")
print(f"Casillas intermedias: N(a3) ∩ N(e3) = {intermedias}")

prob_2pasos = Fraction(0, 1)

print(f"\nCaminos posibles de 2 pasos:")
for k in intermedias:
    c_k, f_k = conv.index(k[0]), int(k[1]) - 1
    grado_k = car_N[c_k][f_k]
    
    p_a3_k = Fraction(1, len(vecinos_a3))
    p_k_e3 = Fraction(1, grado_k)
    p_camino = p_a3_k * p_k_e3
    prob_2pasos += p_camino
    
    print(f"  Camino a3 -> {k} -> e3:")
    print(f"    P(a3 -> {k}) * P({k} -> e3) = {p_a3_k} * {p_k_e3} = {p_camino}")

print("\n" + "-" * 70)
print(f"Probabilidad total en 2 pasos P(X_{{n+2}} = e3 | X_n = a3):")
print(f"  Fracción exacta: {prob_2pasos}")
print(f"  Valor decimal:   {float(prob_2pasos):.8f}")

# Verificación con la potencia de la matriz P^2
idx_a3 = conv.index("a") * 8 + 2
idx_e3 = conv.index("e") * 8 + 2
p2_matriz = sum(P[idx_a3][m] * P[m][idx_e3] for m in range(64))
print(f"  Verificación con entrada (P^2)[a3, e3]: {p2_matriz:.8f} (Coincidencia: {abs(p2_matriz - float(prob_2pasos)) < 1e-9})")


PUNTO 2.4: CÁLCULO DE P(X_{n+2} = e3 | X_n = a3) EN 2 PASOS
Vecinos de a3: N(a3) = ['b1', 'b5', 'c2', 'c4'] (|N(a3)| = 4)
Vecinos de e3: N(e3) = ['c2', 'c4', 'd1', 'd5', 'f1', 'f5', 'g2', 'g4'] (|N(e3)| = 8)
Casillas intermedias: N(a3) ∩ N(e3) = ['c2', 'c4']

Caminos posibles de 2 pasos:
  Camino a3 -> c2 -> e3:
    P(a3 -> c2) * P(c2 -> e3) = 1/4 * 1/6 = 1/24
  Camino a3 -> c4 -> e3:
    P(a3 -> c4) * P(c4 -> e3) = 1/4 * 1/8 = 1/32

----------------------------------------------------------------------
Probabilidad total en 2 pasos P(X_{n+2} = e3 | X_n = a3):
  Fracción exacta: 7/96
  Valor decimal:   0.07291667
  Verificación con entrada (P^2)[a3, e3]: 0.07291667 (Coincidencia: True)


---
### 2.5) Implementación de Funciones Generales y Validación
> **Objetivo:** Diseñar funciones generales que permitan calcular cualquier probabilidad de la Cadena de Markov:
> 1. `prob_interseccion`: Calcula probabilidades conjuntas de trayectorias con intersecciones como $P(X_1=s_1, X_2=s_2, \dots)$.
> 2. `prob_condicional`: Calcula probabilidades condicionales como $P(A \mid B)$, utilizando el principio formal de la probabilidad condicional:
> $$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$
> mediante llamadas a `prob_interseccion` para el numerador $A \cup B$ y para el denominador $B$.


In [10]:
# ==============================================================================
# PUNTO 2.5: FUNCIONES GENERALES PARA CÁLCULO DE PROBABILIDADES
# ==============================================================================
from fractions import Fraction

def potencia_matriz_exacta(M_frac, exponente):
    """Calcula M^k de forma exacta utilizando fracciones."""
    n = len(M_frac)
    # Matriz identidad
    res = [[Fraction(1 if i == j else 0, 1) for j in range(n)] for i in range(n)]
    base = M_frac
    exp = exponente
    while exp > 0:
        if exp % 2 == 1:
            res = [[sum(res[i][m] * base[m][j] for m in range(n)) for j in range(n)] for i in range(n)]
        base = [[sum(base[i][m] * base[m][j] for m in range(n)) for j in range(n)] for i in range(n)]
        exp //= 2
    return res

# Matriz P con fracciones exactas para cálculos sin error de redondeo
P_frac = [[Fraction(0, 1) for _ in range(64)] for _ in range(64)]
for c in range(8):
    for f in range(8):
        x = c * 8 + f
        for nc, nf in N[c][f]:
            y = nc * 8 + nf
            P_frac[x][y] = Fraction(1, car_N[c][f])

def casilla_a_idx(casilla_str):
    return conv.index(casilla_str[0]) * 8 + (int(casilla_str[1]) - 1)

def prob_interseccion(eventos, x0="a1", verbose=False):
    """
    Calcula la probabilidad conjunta P(X_{t1}=s1, X_{t2}=s2, ..., X_{tm}=sm).
    eventos: diccionario {tiempo: casilla} o lista [s1, s2, ...] (inicia en t=1).
    x0: casilla inicial en t=0 (por defecto 'a1'). Si se pasa None, no se asume X_0 fijo.
    """
    if isinstance(eventos, (list, tuple)):
        dict_ev = {i + 1: s for i, s in enumerate(eventos)}
    else:
        dict_ev = dict(eventos)
        
    if 0 not in dict_ev and x0 is not None:
        dict_ev[0] = x0
        
    tiempos = sorted(dict_ev.keys())
    if len(tiempos) == 0:
        return Fraction(1, 1)
        
    prob = Fraction(1, 1)
    if verbose:
        print(f"Calculando probabilidad conjunta para: {dict_ev}")
        
    for idx in range(len(tiempos) - 1):
        t_ant, t_sig = tiempos[idx], tiempos[idx + 1]
        s_ant, s_sig = dict_ev[t_ant], dict_ev[t_sig]
        gap = t_sig - t_ant
        
        if gap == 1:
            c_ant, f_ant = conv.index(s_ant[0]), int(s_ant[1]) - 1
            es_legal = (conv.index(s_sig[0]), int(s_sig[1]) - 1) in N[c_ant][f_ant]
            p_paso = Fraction(1, car_N[c_ant][f_ant]) if es_legal else Fraction(0, 1)
        else:
            P_gap = potencia_matriz_exacta(P_frac, gap)
            p_paso = P_gap[casilla_a_idx(s_ant)][casilla_a_idx(s_sig)]
            
        if verbose:
            print(f"  Paso t={t_ant} ({s_ant}) -> t={t_sig} ({s_sig}) [salto de {gap}]: {p_paso}")
            
        prob *= p_paso
        if prob == 0:
            break
            
    return prob

def prob_condicional(objetivo, condicion, x0="a1", verbose=False):
    """
    Calcula P(Objetivo | Condicion) = P(Objetivo ∩ Condicion) / P(Condicion).
    Utiliza directamente la función prob_interseccion.
    """
    if isinstance(objetivo, (list, tuple)):
        obj_dict = {i + 1: s for i, s in enumerate(objetivo)}
    else:
        obj_dict = dict(objetivo)
        
    if isinstance(condicion, (list, tuple)):
        cond_dict = {i + 1: s for i, s in enumerate(condicion)}
    else:
        cond_dict = dict(condicion)
        
    # La intersección es la unión de ambos diccionarios de eventos
    interseccion_dict = dict(cond_dict)
    interseccion_dict.update(obj_dict)
    
    if verbose:
        print("\nEvaluando probabilidad condicional P(A | B):")
        print(f"  Evento A (Objetivo):   {obj_dict}")
        print(f"  Evento B (Condición):  {cond_dict}")
        print(f"  Intersección A ∩ B:    {interseccion_dict}")
        
    p_num = prob_interseccion(interseccion_dict, x0=x0, verbose=verbose)
    p_den = prob_interseccion(cond_dict, x0=x0, verbose=verbose)
    
    if p_den == 0:
        raise ValueError("Error: El evento condicionante tiene probabilidad 0.")
        
    p_resultado = p_num / p_den
    if verbose:
        print(f"  Resultado: P(A ∩ B) / P(B) = {p_num} / {p_den} = {p_resultado}")
        
    return p_resultado

# ==============================================================================
# VALIDACIÓN AUTOMÁTICA DE TODOS LOS RESULTADOS DEL PUNTO 2
# ==============================================================================
print("=" * 75)
print("VALIDACIÓN AUTOMÁTICA DE LAS 4 PROBABILIDADES CALCULADAS")
print("=" * 75)

# Caso 1: P(X_1 = c2)
res1 = prob_interseccion(["c2"], x0="a1")
print(f"1. prob_interseccion(['c2'], x0='a1'):")
print(f"   Calculado: {res1} ({float(res1):.4f}) | Esperado: 1/2 -> {'CORRECTO' if res1 == Fraction(1, 2) else 'ERROR'}")

# Caso 2: P(X_1=c2, X_2=d4, X_3=e6, X_4=c7)
res2 = prob_interseccion(["c2", "d4", "e6", "c7"], x0="a1")
print(f"\n2. prob_interseccion(['c2', 'd4', 'e6', 'c7'], x0='a1'):")
print(f"   Calculado: {res2} ({float(res2):.8f}) | Esperado: 1/768 -> {'CORRECTO' if res2 == Fraction(1, 768) else 'ERROR'}")

# Caso 3: P(X_3=e2, X_2=d4 | X_1=b3)
res3 = prob_condicional(objetivo={2: "d4", 3: "e2"}, condicion={1: "b3"}, x0="a1")
print(f"\n3. prob_condicional(objetivo={{2: 'd4', 3: 'e2'}}, condicion={{1: 'b3'}}, x0='a1'):")
print(f"   Calculado: {res3} ({float(res3):.6f}) | Esperado: 1/48 -> {'CORRECTO' if res3 == Fraction(1, 48) else 'ERROR'}")

# Caso 4: P(X_{n+2}=e3 | X_n=a3)
# Al ser homogénea en el tiempo, tomamos n=0 sin pérdida de generalidad
res4 = prob_condicional(objetivo={2: "e3"}, condicion={0: "a3"}, x0="a3")
print(f"\n4. prob_condicional(objetivo={{2: 'e3'}}, condicion={{0: 'a3'}}, x0='a3'):")
print(f"   Calculado: {res4} ({float(res4):.8f}) | Esperado: 7/96 -> {'CORRECTO' if res4 == Fraction(7, 96) else 'ERROR'}")


VALIDACIÓN AUTOMÁTICA DE LAS 4 PROBABILIDADES CALCULADAS
1. prob_interseccion(['c2'], x0='a1'):
   Calculado: 1/2 (0.5000) | Esperado: 1/2 -> CORRECTO

2. prob_interseccion(['c2', 'd4', 'e6', 'c7'], x0='a1'):
   Calculado: 1/768 (0.00130208) | Esperado: 1/768 -> CORRECTO

3. prob_condicional(objetivo={2: 'd4', 3: 'e2'}, condicion={1: 'b3'}, x0='a1'):
   Calculado: 1/48 (0.020833) | Esperado: 1/48 -> CORRECTO

4. prob_condicional(objetivo={2: 'e3'}, condicion={0: 'a3'}, x0='a3'):
   Calculado: 7/96 (0.07291667) | Esperado: 7/96 -> CORRECTO
